# cet-perch — fine-tuning Perch V2 on cetacean audio (TensorFlow)

This notebook fine-tunes the last 1–2 blocks of Perch V2's EfficientNet-B3 backbone
on the curated cetacean training set. It follows best practices for transfer
learning on foundation models:

1. **Two-stage training** — first train a new head on top of the frozen backbone
   to convergence, *then* unfreeze the last block(s) and continue with a very
   low learning rate. Mixing randomly-initialised heads with unfrozen pretrained
   layers in one stage causes large gradient updates that destroy pretrained
   features.
2. **BatchNorm safety** — when unfreezing layers, BN layers are kept in inference
   mode (`training=False` at call time) so their running statistics are not
   corrupted by the small fine-tuning batch sizes.
3. **Conservative unfreezing** — start with one block, evaluate, then decide
   whether to unfreeze a second.
4. **Domain-adversarial head** — a second head predicts dataset ID with gradient
   reversal, pushing the backbone to produce dataset-invariant features.
5. **Class-balanced sampling + class-weighted loss** — both at once (belt and
   braces) because of severe class imbalance.
6. **Two ablation runs** — λ_adv=0 (no adversarial) and λ_adv>0 (with), so we
   can isolate whether the adversarial head actually helps.

The notebook is structured so each section runs independently if you have the
preceding artefacts cached.

> **Prerequisite**: an audio cache `X_audio.npy` of shape `(N, 160000)` float32
> and aligned `meta_train.parquet` with `audio_row`, `group_key`, `dataset`,
> `coarse_class` columns. Build these from your curated dataset before running
> this notebook.


## 0. Imports and reproducibility

In [1]:
import sys, json, time, pickle, warnings, random
from pathlib import Path
from collections import Counter, defaultdict

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow import keras

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, balanced_accuracy_score, accuracy_score,
)

from perch_hoplite.zoo import model_configs

warnings.filterwarnings('ignore')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
print(f"TF version : {tf.__version__}")

print(f"GPUs found : {len(gpus)}")
for g in gpus:
    print(f"  {g}")
    try: tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError as e: print(f"  could not set memory growth: {e}")

2026-05-19 10:44:39.111247: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-19 10:44:39.162422: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-19 10:44:40.511928: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TF version : 2.20.0
GPUs found : 1
  PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


## 1. Paths

Everything is keyed off `OUT_DIR`. The notebook produces:
- `cet_perch_stage1.keras` — model after stage 1 (head only)
- `cet_perch_stage2_no_adv.keras` — stage 2 ablation (no adversarial head)
- `cet_perch_stage2_adv.keras` — stage 2 with adversarial head
- `training_history.json` — loss/metric curves per run
- `test_predictions_*.parquet` — per-sample predictions for inspection


In [ ]:
# ── Inputs ────────────────────────────────────────────────────────────────────
CURATION_CSV = Path('/data2/mromaniuc/cet-det/cet_perch/training_set_composition.csv')
META_PARQUET = Path('/data2/mromaniuc/cet-det/alltogether/preliminary/mlp_runs/meta_all_with_taxonomy.parquet')

# Audio cache built by the data-prep notebook (NOT this one)
AUDIO_CACHE  = Path('/data2/mromaniuc/cet-det/cet_perch/X_audio.npy')
AUDIO_META   = Path('/data2/mromaniuc/cet-det/cet_perch/meta_train.parquet')

# ── Outputs ───────────────────────────────────────────────────────────────────
OUT_DIR = Path('/data2/mromaniuc/cet-det/cet_perch/runs')
OUT_DIR.mkdir(exist_ok=True, parents=True)

for label, p in [('curation csv', CURATION_CSV), ('meta parquet', META_PARQUET),
                 ('audio cache', AUDIO_CACHE), ('audio meta', AUDIO_META)]:
    print(f"  {label:15s}  {'OK' if p.exists() else 'MISSING'}  {p}")


## 2. Load curated training data

The audio array is mmap'd — only rows you actually touch are loaded into RAM.


In [ ]:
X_audio = np.load(AUDIO_CACHE, mmap_mode='r')
meta    = pd.read_parquet(AUDIO_META)

print(f"X_audio : {X_audio.shape}  dtype={X_audio.dtype}")
print(f"meta    : {len(meta):,} rows  cols={list(meta.columns)}")

assert len(meta) == X_audio.shape[0], \
    f"audio_cache length {X_audio.shape[0]} != meta length {len(meta)}"
assert X_audio.shape[1] == 160_000, f"expected 160000 samples, got {X_audio.shape[1]}"
assert {'audio_row','group_key','dataset','coarse_class'}.issubset(meta.columns), \
    f"missing required columns; have {list(meta.columns)}"
assert (meta['audio_row'].values == np.arange(len(meta))).all(), \
    "audio_row must equal row position in meta"

print("\nClass distribution:")
print(meta['coarse_class'].value_counts().to_string())
print("\nDataset distribution:")
print(meta['dataset'].value_counts().to_string())


## 3. Encode classes and datasets

Rare datasets (<100 samples) are folded into 'other' so the adversarial head
has well-conditioned classes to predict.


In [ ]:
class_le = LabelEncoder().fit(sorted(meta['coarse_class'].unique()))
meta['y_class'] = class_le.transform(meta['coarse_class'])
N_CLASSES = len(class_le.classes_)

print(f"N_CLASSES = {N_CLASSES}")
for i, c in enumerate(class_le.classes_):
    print(f"  {i:2d}  {(meta['y_class']==i).sum():>6,}  {c}")

MIN_DATASET_SIZE = 100
ds_counts = meta['dataset'].value_counts()
rare_datasets = set(ds_counts[ds_counts < MIN_DATASET_SIZE].index)
meta['dataset_for_adv'] = meta['dataset'].where(
    ~meta['dataset'].isin(rare_datasets), 'other'
)
ds_le = LabelEncoder().fit(sorted(meta['dataset_for_adv'].unique()))
meta['y_dataset'] = ds_le.transform(meta['dataset_for_adv'])
N_DATASETS = len(ds_le.classes_)

print(f"\nN_DATASETS (for adversarial head) = {N_DATASETS}")
for i, d in enumerate(ds_le.classes_):
    print(f"  {i:2d}  {(meta['y_dataset']==i).sum():>6,}  {d}")


## 4. Group-aware train / val / test split

80 / 10 / 10 with `group_key` so windows from the same audio file or deployment
never appear in two splits. Not stratified by class — too-small classes would
all land in one split.


In [ ]:
TEST_FRAC, VAL_FRAC = 0.10, 0.10

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=SEED)
trainval_idx, test_idx = next(gss.split(np.arange(len(meta)), groups=meta['group_key']))

gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC/(1-TEST_FRAC), random_state=SEED)
train_pos, val_pos = next(gss2.split(trainval_idx, groups=meta.iloc[trainval_idx]['group_key']))
train_idx = trainval_idx[train_pos]
val_idx   = trainval_idx[val_pos]

print(f"Train : {len(train_idx):>6,}  ({len(train_idx)/len(meta)*100:.1f}%)")
print(f"Val   : {len(val_idx):>6,}  ({len(val_idx)/len(meta)*100:.1f}%)")
print(f"Test  : {len(test_idx):>6,}  ({len(test_idx)/len(meta)*100:.1f}%)")

# Group-leakage check
for a, b, na, nb in [(train_idx, val_idx, 'train', 'val'),
                     (train_idx, test_idx, 'train', 'test'),
                     (val_idx,   test_idx, 'val',   'test')]:
    ga = set(meta.iloc[a]['group_key']); gb = set(meta.iloc[b]['group_key'])
    overlap = ga & gb
    assert not overlap, f"{na}/{nb} group leakage: {len(overlap)} shared groups"
print("\n  No group leakage between splits.")

print("\nPer-split class counts:")
split_df = pd.DataFrame({
    'class': class_le.classes_,
    'train': [(meta.iloc[train_idx]['y_class']==i).sum() for i in range(N_CLASSES)],
    'val':   [(meta.iloc[val_idx]['y_class']  ==i).sum() for i in range(N_CLASSES)],
    'test':  [(meta.iloc[test_idx]['y_class'] ==i).sum() for i in range(N_CLASSES)],
})
print(split_df.to_string(index=False))


## 5. Class weights for the loss

Inverse-frequency with a cap. Uncapped weights for a 144-sample class would be
~80× the largest class's weight; the model would thrash on those few samples.


In [ ]:
train_class_counts = pd.Series(meta.iloc[train_idx]['y_class']).value_counts().sort_index()
n_total = train_class_counts.sum()
raw_weights = n_total / (N_CLASSES * train_class_counts)
cap = raw_weights.min() * 10
class_weights = raw_weights.clip(upper=cap)
class_weights = class_weights / class_weights.mean()
class_weight_dict = {int(i): float(w) for i, w in class_weights.items()}

print("Class weights (capped, mean-normalised):")
for i, c in enumerate(class_le.classes_):
    n = int(train_class_counts.get(i, 0))
    w = class_weight_dict.get(i, 0.0)
    print(f"  {i:2d}  n={n:>5d}  w={w:.3f}   {c}")


## 6. Balanced batch sampler

`tf.data.Dataset.sample_from_datasets` joins per-class sub-datasets with equal
probability. Combined with class-weighted loss this is belt-and-braces: the
sampler ensures rare classes are seen often; the weight ensures they count
appropriately in the loss.


In [ ]:
BATCH_SIZE  = 32
AUDIO_LEN   = 160_000
EPOCH_STEPS = max(1, len(train_idx) // BATCH_SIZE)

def build_dataset(idx_array, *, training, batch_size=BATCH_SIZE):
    audio_rows = meta.iloc[idx_array]['audio_row'].values.astype(np.int64)
    y_class    = meta.iloc[idx_array]['y_class'].values.astype(np.int32)
    y_dataset  = meta.iloc[idx_array]['y_dataset'].values.astype(np.int32)

    def gen_for(positions):
        positions = list(positions)
        random.shuffle(positions)  
        def _gen():
            for pos in positions:

            for pos in positions:
                row = int(audio_rows[pos])
                yield X_audio[row].astype(np.float32), y_class[pos], y_dataset[pos]
        return _gen

    sig = (tf.TensorSpec(shape=(AUDIO_LEN,), dtype=tf.float32),
           tf.TensorSpec(shape=(),          dtype=tf.int32),
           tf.TensorSpec(shape=(),          dtype=tf.int32))

    if training:
        per_class_dss = []
        for c in range(N_CLASSES):
            pos = np.where(y_class == c)[0]
            if len(pos) == 0: continue
            ds = tf.data.Dataset.from_generator(
                gen_for(pos.tolist()), output_signature=sig,
            ).repeat().shuffle(min(2000, len(pos)))
            per_class_dss.append(ds)
        ds = tf.data.Dataset.sample_from_datasets(
            per_class_dss,
            weights=[1.0/len(per_class_dss)] * len(per_class_dss),
            seed=SEED,
        )
    else:
        ds = tf.data.Dataset.from_generator(
            gen_for(list(range(len(idx_array)))), output_signature=sig,
        )

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


print("Building datasets...")
train_ds = build_dataset(train_idx, training=True)
val_ds   = build_dataset(val_idx,   training=False)
test_ds  = build_dataset(test_idx,  training=False)

print("\nSmoke test — one batch from each:")
for name, ds in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    for x, yc, yd in ds.take(1):
        print(f"  {name:5s}  audio={tuple(x.shape)}  y_class={tuple(yc.shape)}  "
              f"y_dataset={tuple(yd.shape)}  amplitude=[{tf.reduce_min(x):.3f}, {tf.reduce_max(x):.3f}]")
        break


## 7. Load Perch V2 and inspect its structure

The Perch V2 model loads as a `tf.saved_model.load()` object wrapped by
`perch_hoplite`. We need direct access to the underlying SavedModel to see its
`trainable_variables` and identify the variables in the last block(s).

The first cell finds the raw SavedModel. The second prints variable names so
you can see how the blocks are named (this is critical — different Perch
versions use different naming conventions).


In [11]:
print("Loading Perch V2 (downloads from Kaggle on first run, then caches)...")
perch_wrapper = model_configs.load_model_by_name('perch_v2')
print(f"  loaded: {type(perch_wrapper).__name__}")

# Perch V2 is wrapped in TaxonomyModelTF; the SavedModel is at .model
raw_model = perch_wrapper.model
print(f"  raw SavedModel @ perch_wrapper.model")

# Get the forward function and embedding key
FORWARD_FN = raw_model.signatures['serving_default']
EMBEDDING_KEY = 'embedding'

print(f"  FORWARD_FN: signatures['serving_default']")
print(f"  EMBEDDING_KEY: '{EMBEDDING_KEY}'")

# Variable counts come from the concrete function, not the _UserObject wrapper
print(f"\n  trainable_variables : {len(FORWARD_FN.trainable_variables)}")
print(f"  variables (total)    : {len(FORWARD_FN.variables)}")

# Verify output structure
print(f"\n  Output keys: {list(FORWARD_FN.structured_outputs.keys())}")
print(f"  Embedding shape: {FORWARD_FN.structured_outputs[EMBEDDING_KEY].shape}")

Loading Perch V2 (downloads from Kaggle on first run, then caches)...


I0000 00:00:1779180531.298274  805275 gpu_device.cc:2020] Created device /device:GPU:0 with 20749 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:3b:00.0, compute capability: 8.9


  loaded: TaxonomyModelTF
  raw SavedModel @ perch_wrapper.model
  FORWARD_FN: signatures['serving_default']
  EMBEDDING_KEY: 'embedding'

  trainable_variables : 0
  variables (total)    : 497

  Output keys: ['spatial_embedding', 'label', 'embedding', 'spectrogram']
  Embedding shape: (None, 1536)


In [13]:
# Sanity check — should see EfficientNet block names
print(f"First 10 variables:")
for v in FORWARD_FN.variables[:10]:
    print(f"  {v.name:60s}  {v.shape}")

print(f"\nLast 10 variables:")
for v in FORWARD_FN.variables[-10:]:
    print(f"  {v.name:60s}  {v.shape}")

First 10 variables:
  batch_stats.embedding_model.backbone.Head_0.BatchNorm_0.mean:0  (1536,)
  batch_stats.embedding_model.backbone.Head_0.BatchNorm_0.var:0  (1536,)
  batch_stats.embedding_model.backbone.MBConv_0.DepthwiseBatchNorm.mean:0  (40,)
  batch_stats.embedding_model.backbone.MBConv_0.DepthwiseBatchNorm.var:0  (40,)
  batch_stats.embedding_model.backbone.MBConv_0.ProjectBatchNorm.mean:0  (24,)
  batch_stats.embedding_model.backbone.MBConv_0.ProjectBatchNorm.var:0  (24,)
  batch_stats.embedding_model.backbone.MBConv_1.DepthwiseBatchNorm.mean:0  (24,)
  batch_stats.embedding_model.backbone.MBConv_1.DepthwiseBatchNorm.var:0  (24,)
  batch_stats.embedding_model.backbone.MBConv_1.ProjectBatchNorm.mean:0  (24,)
  batch_stats.embedding_model.backbone.MBConv_1.ProjectBatchNorm.var:0  (24,)

Last 10 variables:
  params.embedding_model.backbone.MBConv_9.SqueezeAndExcitation_0.Expand.bias:0  (576,)
  params.embedding_model.backbone.MBConv_9.SqueezeAndExcitation_0.Expand.kernel:0  (24, 5

In [15]:
import re
block_names = sorted(set(
    m.group(0)
    for v in FORWARD_FN.variables
    for m in [re.search(r'MBConv_\d+', v.name)]
    if m
))
print(f"All MBConv blocks in model: {block_names}")

All MBConv blocks in model: ['MBConv_0', 'MBConv_1', 'MBConv_10', 'MBConv_11', 'MBConv_12', 'MBConv_13', 'MBConv_14', 'MBConv_15', 'MBConv_16', 'MBConv_17', 'MBConv_18', 'MBConv_19', 'MBConv_2', 'MBConv_20', 'MBConv_21', 'MBConv_22', 'MBConv_23', 'MBConv_24', 'MBConv_25', 'MBConv_3', 'MBConv_4', 'MBConv_5', 'MBConv_6', 'MBConv_7', 'MBConv_8', 'MBConv_9']


In [19]:
# ── Count variables by MBConv block ───────────────────────────────────────────
import re

block_counts = Counter()
for v in FORWARD_FN.variables:
    name = v.name
    m = re.search(r'MBConv_(\d+)', name)
    if m:
        block_counts[f"MBConv_{m.group(1)}"] += 1
    elif name.startswith('batch_stats.') and 'MBConv' not in name:
        block_counts['Head/Stem BN stats'] += 1
    elif name.startswith('params.') and 'MBConv' not in name:
        block_counts['Head/Stem params'] += 1
    else:
        block_counts['other'] += 1

print("Variables grouped by MBConv block:")
for k in sorted(block_counts.keys(), key=lambda x: int(re.search(r'\d+', x).group()) if re.search(r'\d+', x) else -1):
    print(f"  {k:25s} {block_counts[k]}")

mbconv_indices = [int(re.search(r'MBConv_(\d+)', k).group(1)) for k in block_counts if 'MBConv' in k]
last_block = max(mbconv_indices)
total_blocks = last_block + 1

print(f"\nTotal MBConv blocks: {total_blocks}")
print(f"Last block: MBConv_{last_block}")
print(f"\nFor Stage 2:")
print(f"  n_blocks=1 → unfreeze MBConv_{last_block} only")
print(f"  n_blocks=2 → unfreeze MBConv_{last_block-1} + MBConv_{last_block}")

Variables grouped by MBConv block:
  Head/Stem BN stats        4
  Head/Stem params          9
  MBConv_0                  14
  MBConv_1                  14
  MBConv_2                  19
  MBConv_3                  19
  MBConv_4                  19
  MBConv_5                  19
  MBConv_6                  19
  MBConv_7                  19
  MBConv_8                  19
  MBConv_9                  19
  MBConv_10                 19
  MBConv_11                 19
  MBConv_12                 19
  MBConv_13                 19
  MBConv_14                 19
  MBConv_15                 19
  MBConv_16                 19
  MBConv_17                 19
  MBConv_18                 19
  MBConv_19                 19
  MBConv_20                 19
  MBConv_21                 19
  MBConv_22                 19
  MBConv_23                 19
  MBConv_24                 19
  MBConv_25                 19

Total MBConv blocks: 26
Last block: MBConv_25

For Stage 2:
  n_blocks=1 → unfreeze MBConv_25 only

## 8. Find the differentiable forward function

`perch_hoplite.zoo`'s `embed()` method may call `.numpy()` internally, which
breaks the gradient tape. For fine-tuning we need to call the raw SavedModel's
forward function directly — which keeps everything in TF and preserves
gradients.

Different Perch versions store the callable in different places. The cell
below tries common patterns; if none work, inspect the output and pick the
right one manually.


In [20]:
print("Available signatures:")
if hasattr(raw_model, 'signatures'):
    for name, sig in raw_model.signatures.items():
        print(f"\n  {name}")
        try:
            print(f"    inputs:  {sig.structured_input_signature}")
            print(f"    outputs: {list(sig.structured_outputs.keys()) if isinstance(sig.structured_outputs, dict) else sig.structured_outputs}")
        except Exception as e:
            print(f"    (couldn't introspect: {e})")

print("\n__call__ available :", hasattr(raw_model, '__call__'))
print("embed   available :", hasattr(raw_model, 'embed'))

Available signatures:

  serving_default
    inputs:  ((), {'inputs': TensorSpec(shape=(None, 160000), dtype=tf.float32, name='inputs')})
    outputs: ['spatial_embedding', 'label', 'embedding', 'spectrogram']

__call__ available : False
embed   available : False

After inspecting the above, set FORWARD_FN to the right callable.
Common pattern: FORWARD_FN = raw_model.signatures['serving_default']

It should accept a (B, 160000) float32 tensor and return a dict with
an 'embedding' key (or similar) holding a (B, 1536) tensor.


In [22]:
# ── Forward function & embedding key (resolved from inspection above) ──────────
FORWARD_FN   = raw_model.signatures['serving_default']
EMBEDDING_KEY = 'embedding'

# Verify
dummy = tf.constant(np.zeros((1, 160000), dtype=np.float32))
out   = FORWARD_FN(inputs=dummy)
assert out[EMBEDDING_KEY].shape == (1, 1536), f"Unexpected shape: {out[EMBEDDING_KEY].shape}"

print(f"FORWARD_FN   : signatures['serving_default']")
print(f"EMBEDDING_KEY: '{EMBEDDING_KEY}'")
print(f"Embedding shape check: {out[EMBEDDING_KEY].shape} ✓")
print(f"Input key: 'inputs', shape (None, 160000) float32")
print(f"All output keys: {list(out.keys())}")

FORWARD_FN   : signatures['serving_default']
EMBEDDING_KEY: 'embedding'
Embedding shape check: (1, 1536) ✓
Input key: 'inputs', shape (None, 160000) float32
All output keys: ['spatial_embedding', 'label', 'embedding', 'spectrogram']


In [24]:
# ── Once FORWARD_FN is set, verify it works ───────────────────────────────────
AUDIO_LEN = 160000  # 5s @ 32kHz

if FORWARD_FN is not None:
    test_audio = np.zeros((2, AUDIO_LEN), dtype=np.float32)
    test_out = FORWARD_FN(inputs=tf.constant(test_audio))  # keyword arg required
    print(f"Output type    : {type(test_out)}")
    if isinstance(test_out, dict):
        print(f"Output keys    : {list(test_out.keys())}")
        for k, v in test_out.items():
            print(f"  {k:20s} shape={tuple(v.shape)}  dtype={v.dtype}")
    else:
        print(f"Output shape   : {tuple(test_out.shape)}")
    print(f"\nEMBEDDING_KEY = '{EMBEDDING_KEY}'  shape={tuple(test_out[EMBEDDING_KEY].shape)} ✓")
else:
    print("FORWARD_FN not set yet — go back to the previous cell.")

2026-05-19 10:53:53.207926: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_29', 4 bytes spill stores, 4 bytes spill loads

2026-05-19 10:53:53.298339: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_29', 4 bytes spill stores, 4 bytes spill loads

2026-05-19 10:53:53.732378: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_55', 64 bytes spill stores, 64 bytes spill loads

2026-05-19 10:53:54.043466: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_55', 956 bytes spill stores, 956 bytes spill loads

2026-05-19 10:53:56.446967: E external/local_xla/x

Output type    : <class 'dict'>
Output keys    : ['spatial_embedding', 'label', 'embedding', 'spectrogram']
  spatial_embedding    shape=(2, 16, 4, 1536)  dtype=<dtype: 'float32'>
  label                shape=(2, 14795)  dtype=<dtype: 'float32'>
  embedding            shape=(2, 1536)  dtype=<dtype: 'float32'>
  spectrogram          shape=(2, 500, 128)  dtype=<dtype: 'float32'>

EMBEDDING_KEY = 'embedding'  shape=(2, 1536) ✓


## 9. Define the trainable model

```
audio (160000,)
  ↓
Perch V2 backbone  →  embedding (1536,)
  ↓                          ↓
classification head    domain head (with gradient reversal)
  ↓                          ↓
softmax over 10 classes    softmax over N datasets
```

The gradient reversal layer is identity on the forward pass but multiplies the
gradient by `-λ` on the backward pass. This pushes the backbone to *not* learn
features useful for predicting the dataset — i.e. dataset-invariant features.


In [27]:
class GradientReversalLayer(tf.keras.layers.Layer):
    """Identity forward; multiplies gradient by -lambda backward.

    `lambda_value` is a tf.Variable so we can change it during training.
    """
    def __init__(self, lambda_value=0.0, **kwargs):
        super().__init__(**kwargs)
        self.lambda_value = tf.Variable(
            float(lambda_value), trainable=False, name='grl_lambda', dtype=tf.float32,
        )

    @tf.custom_gradient
    def _grad_reverse(self, x):
        lam = self.lambda_value
        def grad(dy):
            return -lam * dy
        return tf.identity(x), grad

    def call(self, inputs):
        return self._grad_reverse(inputs)

    def get_config(self):
        cfg = super().get_config()
        cfg['lambda_value'] = float(self.lambda_value.numpy())
        return cfg


class PerchBackbone(tf.keras.layers.Layer):
    """Calls Perch's forward function. Keeps gradients flowing.

    IMPORTANT: pass training=False here even during fine-tuning. EfficientNet-B3
    BatchNorm layers inside the SavedModel will stay in inference mode, using
    their stored moving means/variances rather than re-computing per-batch stats.
    This is the TF transfer-learning best practice.
    """
    def __init__(self, forward_fn, embedding_key='embedding', **kwargs):
        super().__init__(**kwargs)
        self.forward_fn = forward_fn
        self.embedding_key = embedding_key

    def call(self, audio, training=False):
        outputs = self.forward_fn(inputs=audio)  # ← keyword required
        if isinstance(outputs, dict):
            emb = outputs[self.embedding_key]
        else:
            emb = outputs
        if emb.shape.rank == 3:
            emb = tf.reduce_mean(emb, axis=1)
        return emb


def build_cet_perch(forward_fn, raw_model, *, n_classes, n_datasets,
                    use_adversarial=True, lambda_adv=0.3,
                    embedding_key='embedding', head_dropout=0.3):
    """Build the full cet-perch model.

    Returns a Keras Model with:
      - output 'class_logits' : (B, n_classes)
      - output 'domain_logits': (B, n_datasets)  [only if use_adversarial]

    The raw SavedModel is stashed on the model as `_perch_raw_model` so the
    freeze/unfreeze helpers below can find and manipulate its variables.
    """
    inputs = tf.keras.Input(shape=(AUDIO_LEN,), dtype=tf.float32, name='audio')

    backbone = PerchBackbone(forward_fn, embedding_key=embedding_key, name='perch_backbone')
    embedding = backbone(inputs, training=False)

    # Classification head
    x = tf.keras.layers.Dropout(head_dropout, name='cls_dropout1')(embedding)
    x = tf.keras.layers.Dense(256, activation='relu', name='cls_dense1')(x)
    x = tf.keras.layers.Dropout(head_dropout, name='cls_dropout2')(x)
    class_logits = tf.keras.layers.Dense(n_classes, name='class_logits')(x)

    outputs = {'class_logits': class_logits}

    # Domain (adversarial) head
    if use_adversarial:
        grl = GradientReversalLayer(lambda_value=lambda_adv, name='grl')
        x_adv = grl(embedding)
        x_adv = tf.keras.layers.Dense(256, activation='relu', name='dom_dense1')(x_adv)
        x_adv = tf.keras.layers.Dropout(head_dropout, name='dom_dropout1')(x_adv)
        domain_logits = tf.keras.layers.Dense(n_datasets, name='domain_logits')(x_adv)
        outputs['domain_logits'] = domain_logits

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='cet_perch')
    model._perch_raw_model = raw_model
    return model

print("Model builder defined. Build with:")
print("  model = build_cet_perch(FORWARD_FN, raw_model,")
print("              n_classes=N_CLASSES, n_datasets=N_DATASETS,")
print("              use_adversarial=False, embedding_key=EMBEDDING_KEY)")


Model builder defined. Once FORWARD_FN/EMBEDDING_KEY are set, build with:
  model = build_cet_perch(FORWARD_FN, raw_model,
              n_classes=N_CLASSES, n_datasets=N_DATASETS,
              use_adversarial=False, embedding_key=EMBEDDING_KEY)


## 10. Stage 1: train head only (backbone fully frozen)

The TF best-practice: **train the head to convergence first**, before unfreezing
anything. This prevents the random head weights from causing large gradients
that destroy the pretrained backbone.

- All Perch backbone variables: frozen
- New classification head: trainable
- Adversarial head: disabled (we'll add it in stage 2)
- Learning rate: `1e-3` (standard for new layers)


In [28]:
def freeze_backbone(model):
    for v in model._perch_raw_model.variables:
        v._trainable = False  # SavedModel variables have no public setter, this is unavoidable
    for layer in model.layers:
        if isinstance(layer, PerchBackbone):
            layer.trainable = False
    model._backbone_frozen = True
    print(f"  frozen {len(model._perch_raw_model.variables)} backbone variables")
    

STAGE1_EPOCHS   = 30
STAGE1_LR       = 1e-3
STAGE1_PATIENCE = 5

def compile_stage1(model):
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=STAGE1_LR, weight_decay=1e-5),
        loss={'class_logits': tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)},
        metrics={'class_logits': ['sparse_categorical_accuracy']},
    )

def fit_stage1(model, train_ds, val_ds):
    cbs = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(OUT_DIR / 'cet_perch_stage1.keras'),
            monitor='val_class_logits_sparse_categorical_accuracy',
            mode='max', save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_class_logits_sparse_categorical_accuracy',
            mode='max', patience=STAGE1_PATIENCE, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ]
    # tf.data yields (audio, y_class, y_dataset); strip y_dataset for stage 1
    def strip(audio, y_class, y_dataset):
        return audio, y_class
    h = model.fit(
        train_ds.map(strip),
        validation_data=val_ds.map(strip),
        steps_per_epoch=EPOCH_STEPS,
        epochs=STAGE1_EPOCHS,
        callbacks=cbs,
        class_weight={'class_logits': class_weight_dict}
        verbose=1,
    )
    return h.history

print("Stage 1 helpers defined. To run:")
print("  model = build_cet_perch(FORWARD_FN, raw_model, n_classes=N_CLASSES,")
print("                          n_datasets=N_DATASETS, use_adversarial=False,")
print("                          embedding_key=EMBEDDING_KEY)")
print("  freeze_backbone(model)")
print("  compile_stage1(model)")
print("  stage1_history = fit_stage1(model, train_ds, val_ds)")


Stage 1 helpers defined. To run:
  model = build_cet_perch(FORWARD_FN, raw_model, n_classes=N_CLASSES,
                          n_datasets=N_DATASETS, use_adversarial=False,
                          embedding_key=EMBEDDING_KEY)
  freeze_backbone(model)
  compile_stage1(model)
  stage1_history = fit_stage1(model, train_ds, val_ds)


## 11. Stage 2: unfreeze the last block(s)

After stage 1 converges, unfreeze the **last 1 or 2 blocks** of EfficientNet-B3
and continue with `1e-5` learning rate — 100× lower than stage 1 — because we're
modifying pretrained weights and want incremental updates only.

### How many blocks to unfreeze?
EfficientNet-B3 block sizes:
- Block 7: ~1.5M params → with 12k samples, ~8 samples/param. **Safe — start here.**
- Block 6+7: ~3.5M params → ~3 samples/param. Risky; only if block 7 alone plateaus.

### What about BatchNorm in the unfrozen blocks?
- **Conv kernels, BN gamma, BN beta**: trainable.
- **BN moving_mean and moving_variance**: kept frozen (we set `training=False`
  on the backbone call). This is critical — corrupting moving stats with small
  fine-tuning batches is a known failure mode.

### Two ablation runs
- **Variant A**: no adversarial head (pure fine-tuning) — the ablation
- **Variant B**: with adversarial head (λ_adv=0.3) — the experimental condition


In [29]:
UNFREEZE_BLOCKS = 1
STAGE2_EPOCHS   = 30
STAGE2_LR       = 1e-5
STAGE2_PATIENCE = 5

def unfreeze_last_blocks(model, n_blocks):
    """Unfreeze the highest-numbered n_blocks MBConv blocks.

    Perch V2 uses MBConv_0 … MBConv_25 (26 blocks total).
    BN moving stats (batch_stats.*) stay frozen regardless — only
    conv kernels and BN gamma/beta (params.*) become trainable.
    """
    last_block_idx = 25
    blocks_to_unfreeze = [f"MBConv_{last_block_idx - i}" for i in range(n_blocks)]

    n_unfrozen = 0
    n_bn_frozen = 0
    unfrozen_names = []

    for v in model._perch_raw_model.variables:
        name = v.name
        is_in_target = any(f".{b}." in name for b in blocks_to_unfreeze)
        is_bn_stat   = name.startswith('batch_stats.')

        if is_in_target and not is_bn_stat:
            v._trainable = True
            n_unfrozen += 1
            unfrozen_names.append(name)
        else:
            v._trainable = False
            if is_bn_stat and is_in_target:
                n_bn_frozen += 1

    for layer in model.layers:
        if isinstance(layer, PerchBackbone):
            layer.trainable = True

    print(f"Unfreezing blocks: {blocks_to_unfreeze}")
    print(f"Unfroze {n_unfrozen} variables  |  BN moving stats kept frozen: {n_bn_frozen}")
    if unfrozen_names:
        print(f"\nFirst 5 unfrozen variables:")
        for n in unfrozen_names[:5]:
            print(f"  {n}")
        if len(unfrozen_names) > 5:
            print(f"  ... and {len(unfrozen_names)-5} more")
    return n_unfrozen


def compile_stage2(model, *, use_adversarial, lambda_adv=0.3, lambda_dom_loss=0.3):
    losses  = {'class_logits': tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)}
    weights = {'class_logits': 1.0}
    metrics = {'class_logits': ['sparse_categorical_accuracy']}
    if use_adversarial:
        losses['domain_logits']  = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        weights['domain_logits'] = lambda_dom_loss
        metrics['domain_logits'] = ['sparse_categorical_accuracy']
        for layer in model.layers:
            if isinstance(layer, GradientReversalLayer):
                layer.lambda_value.assign(lambda_adv)
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=STAGE2_LR, weight_decay=1e-5),
        loss=losses, loss_weights=weights, metrics=metrics,
    )


def fit_stage2(model, train_ds, val_ds, *, use_adversarial, run_name):
    ckpt = str(OUT_DIR / f'cet_perch_stage2_{run_name}.keras')
    cbs = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=ckpt,
            monitor='val_class_logits_sparse_categorical_accuracy',
            mode='max', save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_class_logits_sparse_categorical_accuracy',
            mode='max', patience=STAGE2_PATIENCE, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ]
    if use_adversarial:
        def adapt(audio, y_class, y_dataset):
            return audio, {'class_logits': y_class, 'domain_logits': y_dataset}
    else:
        def adapt(audio, y_class, y_dataset):
            return audio, {'class_logits': y_class}
    # class_weight not supported with dict outputs; balanced sampler covers it
    h = model.fit(
        train_ds.map(adapt),
        validation_data=val_ds.map(adapt),
        steps_per_epoch=EPOCH_STEPS,
        epochs=STAGE2_EPOCHS,
        callbacks=cbs,
        verbose=1,
    )
    return h.history

print("Stage 2 helpers defined.")

Stage 2 helpers defined.


## 12. Run training

Execute these cells **one at a time**, checking each result before moving on.
Do not run them as one big batch — if stage 1 doesn't converge, stage 2 won't fix it.


In [ ]:
# ── Stage 1: head-only training ───────────────────────────────────────────────
# model = build_cet_perch(FORWARD_FN, raw_model,
#                         n_classes=N_CLASSES, n_datasets=N_DATASETS,
#                         use_adversarial=False, embedding_key=EMBEDDING_KEY)
# freeze_backbone(model)
# compile_stage1(model)
# trainable_p = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
# print(f"Stage 1: {trainable_p:,} trainable params")
# stage1_history = fit_stage1(model, train_ds, val_ds)
# print(f"\nStage 1 best val acc: {max(stage1_history['val_class_logits_sparse_categorical_accuracy']):.3f}")


In [ ]:
# ── Stage 2 variant A: unfreeze, no adversarial (ablation) ────────────────────
# n_unf = unfreeze_last_blocks(model, n_blocks=UNFREEZE_BLOCKS)
# compile_stage2(model, use_adversarial=False)
# trainable_p = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
# print(f"Stage 2A: {trainable_p:,} trainable params (was {n_unf} backbone variables + heads)")
# stage2_no_adv_history = fit_stage2(model, train_ds, val_ds, use_adversarial=False, run_name='no_adv')
# print(f"\nStage 2A best val acc: {max(stage2_no_adv_history['val_class_logits_sparse_categorical_accuracy']):.3f}")


In [ ]:
# ── Stage 2 variant B: with adversarial head ──────────────────────────────────
# # Build fresh model so we have a clean GRL + domain head
# model_adv = build_cet_perch(FORWARD_FN, raw_model,
#                             n_classes=N_CLASSES, n_datasets=N_DATASETS,
#                             use_adversarial=True, lambda_adv=0.3,
#                             embedding_key=EMBEDDING_KEY)
#
# # Initialise classification head from stage 1 to avoid losing those gains
# stage1_model = tf.keras.models.load_model(
#     OUT_DIR / 'cet_perch_stage1.keras',
#     custom_objects={'GradientReversalLayer': GradientReversalLayer,
#                     'PerchBackbone': PerchBackbone},
# )
# for layer_name in ['cls_dense1', 'class_logits']:
#     model_adv.get_layer(layer_name).set_weights(
#         stage1_model.get_layer(layer_name).get_weights()
#     )
#
# # Initialise domain head with GRL OFF (lambda=0) for a few epochs so it learns
# # to predict the dataset before the backbone starts being adversarially trained.
# # Without this, the domain head is uninformed and the adversarial signal is noise.
# freeze_backbone(model_adv)
# compile_stage2(model_adv, use_adversarial=True, lambda_adv=0.0, lambda_dom_loss=0.3)
# print("Adversarial-head warmup (5 epochs, GRL off)...")
# WARMUP_EPOCHS = 5
# saved_e = STAGE2_EPOCHS
# globals()['STAGE2_EPOCHS'] = WARMUP_EPOCHS
# fit_stage2(model_adv, train_ds, val_ds, use_adversarial=True, run_name='adv_warmup')
# globals()['STAGE2_EPOCHS'] = saved_e
#
# # Now turn GRL on, unfreeze the backbone block(s), train for real
# unfreeze_last_blocks(model_adv, n_blocks=UNFREEZE_BLOCKS)
# compile_stage2(model_adv, use_adversarial=True, lambda_adv=0.3, lambda_dom_loss=0.3)
# stage2_adv_history = fit_stage2(model_adv, train_ds, val_ds, use_adversarial=True, run_name='adv')
# print(f"\nStage 2B best val acc: {max(stage2_adv_history['val_class_logits_sparse_categorical_accuracy']):.3f}")


## 13. Evaluate on the held-out test set

The headline number is **macro-F1**, which weighs all classes equally regardless
of how many samples each has.


In [ ]:
def evaluate(model, test_ds):
    y_true, y_pred, y_proba = [], [], []
    for audio, y_class, y_dataset in test_ds:
        out = model(audio, training=False)
        logits = out['class_logits'] if isinstance(out, dict) else out
        proba = tf.nn.softmax(logits, axis=-1).numpy()
        y_true.append(y_class.numpy())
        y_pred.append(proba.argmax(axis=-1))
        y_proba.append(proba)
    return (np.concatenate(y_true), np.concatenate(y_pred), np.concatenate(y_proba))

def report(y_true, y_pred, name):
    print(f"\n{'='*70}\n  {name}\n{'='*70}")
    print(f"  accuracy    : {accuracy_score(y_true, y_pred):.3f}")
    print(f"  balanced acc: {balanced_accuracy_score(y_true, y_pred):.3f}")
    print(f"  macro F1    : {f1_score(y_true, y_pred, average='macro'):.3f}")
    print(f"  weighted F1 : {f1_score(y_true, y_pred, average='weighted'):.3f}")
    print()
    print(classification_report(
        y_true, y_pred,
        labels=list(range(N_CLASSES)),
        target_names=list(class_le.classes_),
        digits=3, zero_division=0,
    ))

# ── Run after both stage-2 variants finish ────────────────────────────────────
# y_true_a, y_pred_a, _ = evaluate(model,     test_ds)
# y_true_b, y_pred_b, _ = evaluate(model_adv, test_ds)
# report(y_true_a, y_pred_a, 'cet-perch — no adversarial')
# report(y_true_b, y_pred_b, 'cet-perch — with adversarial (λ=0.3)')


## 14. Notes and troubleshooting

**Stage 1 sanity**: val accuracy should rise monotonically for ~10–20 epochs then
plateau. Plateauing at <0.5 macro-F1 means either the audio cache is corrupted,
the labels are misaligned, or Perch's embeddings really aren't separable for your
taxonomy (unlikely given prior LODO results).

**Stage 2 sanity**: val accuracy should *improve over stage 1* within the first
5–10 epochs. If it immediately drops, the learning rate is too high — divide by 10
and retry. If train accuracy stalls while val moves, you may have a class-weight
or sampler bug.

**`loss=nan` during stage 2**: this is almost always BatchNorm moving stats being
updated when they shouldn't be. Verify that `training=False` is being passed to
the `PerchBackbone` layer, and that the unfreeze code excluded `moving_mean` and
`moving_variance`. If still nan, drop LR by another 10×.

**Adversarial training**: domain-head accuracy should *decrease* over time as the
backbone learns to confuse it. Starting around 0.6 (datasets are obviously
different) and drifting toward 1/N_DATASETS (~0.07 for ~14 datasets) means the
adversarial signal is working. If domain accuracy stays high or class accuracy
collapses, drop λ_adv from 0.3 → 0.1.

**Class accuracy slightly worse on variant B than variant A?** That's expected
and fine — the goal of the adversarial head is improvement on **LODO** (cross-dataset
generalisation), not on within-dataset val accuracy. The LODO benchmark is where
variant B should win, if it works.